# Pneumonia Two-Stage Training — Google Colab (full run)

End-to-end notebook for the paper reproduction:

- **Stage 1 (Detection):** Normal vs Pneumonia — PneumoNet-v1, PneumoNet-v2, VGG-16, Inception-v3
- **Stage 2 (Classification):** Bacterial vs Viral — PneumoNet-v1, MobileNet-v2, Inception-v3 (Xception substitute)

**Paper targets:** Stage 1 **98.03%** / AUC **0.97** · Stage 2 **91.93%** / AUC **0.91**

## What this notebook does

1. Mount Drive, clone/update the repo, install the package
2. Verify the Stage 1 / Stage 2 data on Drive (rebuild it from a raw `chest_xray` if needed)
3. Build a fast 224px cache on local disk
4. Sanity check
5. Train the remaining **Stage 1** models (skips anything already on Drive)
6. Train the three **Stage 2** models
7. Evaluate each model, both ensembles, and the full two-stage pipeline

## Prerequisites

- Runtime → Change runtime type → **T4 GPU**
- Drive contains `PneumoniaProject/data/stage1` and `PneumoniaProject/data/stage2`
  (already produced by `colab_stage1_training.ipynb`), **or** a raw `chest_xray/` folder / `chest_xray.zip` in `PneumoniaProject/`
- The speedups in `src/pneumonia` (AMP, pinned dataloaders, fast class weights)
  must be **committed and pushed** to GitHub — this notebook clones the repo.

## Stage 1 vs Stage 2 ordering

Stage 2 does **not** depend on Stage 1: it is a separate binary classifier trained only on pneumonia images.
They can be trained in any order. This notebook runs Stage 1 first (primary result), then Stage 2.

## Disconnect behaviour

Epoch checkpoints are written to fast **local disk** and only `final_model.pt` is copied to Drive.
This avoids hundreds of MB of Drive writes per epoch. If the session dies mid-model, that one model
retrains (≈10–20 min); completed models are skipped automatically.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DRIVE_DIR = '/content/drive/MyDrive/PneumoniaProject'
CKPT_DRIVE = os.path.join(PROJECT_DRIVE_DIR, 'checkpoints')
DATA_DRIVE = os.path.join(PROJECT_DRIVE_DIR, 'data')

for d in (PROJECT_DRIVE_DIR, CKPT_DRIVE, DATA_DRIVE):
    os.makedirs(d, exist_ok=True)

print('Drive project:', PROJECT_DRIVE_DIR)
print('Checkpoints :', CKPT_DRIVE)
print('Data        :', DATA_DRIVE)


## 2. Clone or update the repository

The training code lives on the `colab-full-training` branch. Change `BRANCH`
below if you push it somewhere else.

In [ ]:
REPO_URL = 'https://github.com/peas-rip/IEEE-Projects.git'
BRANCH   = 'colab-full-training'
REPO_ROOT = '/content/IEEE-Projects'
PROJECT_DIR = '/content/IEEE-Projects/ML Projects/Pnemonia Treatment'

import os

if not os.path.exists(REPO_ROOT):
    !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}

os.chdir(PROJECT_DIR)
os.environ['PYTHONPATH'] = os.path.join(PROJECT_DIR, 'src')
print('Branch     :', BRANCH)
print('Working dir:', os.getcwd())
!ls src/pneumonia configs scripts


## 3. Install dependencies (torch is preinstalled on Colab)

`pip install -e .` makes the `pneumonia` CLI available to shell subprocesses,
which fixes `No module named pneumonia` during evaluation.


In [ ]:
!pip install -q scikit-learn pyyaml pillow matplotlib tqdm
!pip install -q -e .

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (set Runtime -> T4 GPU!)')


## 4. Runtime checks (GPU + speedups present?)

In [ ]:
import inspect
import os

import torch

if not torch.cuda.is_available():
    raise SystemExit('CUDA is not available. Runtime -> Change runtime type -> T4 GPU.')

torch.backends.cudnn.benchmark = True
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

from pneumonia import train as _train

_HAS_AMP = 'autocast' in inspect.getsource(_train.train_model)
print('Mixed precision (AMP) present in repo:', _HAS_AMP)
if not _HAS_AMP:
    print('WARNING: the cloned repo does not contain the speedups.')
    print('         Commit + push the updated src/ first, or training will be ~2x slower.')


## 5. Verify (or rebuild) the Stage 1 / Stage 2 data on Drive

Self-contained: handles a raw `chest_xray/` folder **or** a `.zip`, and both the
nested (`PNEUMONIA/BACTERIA/`) and flat (`BACTERIA_*.jpeg`) label layouts.


In [ ]:
import shutil
import zipfile
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
DATA = Path(DATA_DRIVE)


def _is_img(p):
    return p.is_file() and p.suffix.lower() in IMG_EXTS


def _label_from_name(name):
    n = name.lower()
    if 'bacteria' in n:
        return 'BACTERIAL'
    if 'virus' in n or 'viral' in n:
        return 'VIRAL'
    return None


def _collect_pneumonia(pneu_dir):
    bact, vir = [], []
    pneu_dir = Path(pneu_dir)
    if not pneu_dir.is_dir():
        return bact, vir
    for p in pneu_dir.rglob('*'):
        if not _is_img(p):
            continue
        parent = p.parent.name.lower()
        if parent in {'bacteria', 'bacterial'}:
            bact.append(p)
            continue
        if parent in {'virus', 'viral'}:
            vir.append(p)
            continue
        lab = _label_from_name(p.name)
        if lab == 'BACTERIAL':
            bact.append(p)
        elif lab == 'VIRAL':
            vir.append(p)
    return bact, vir


def _locate_base(src):
    src = Path(src)
    if (src / 'train').is_dir() and (src / 'test').is_dir():
        return src
    if (src / 'chest_xray').is_dir():
        return src / 'chest_xray'
    for child in src.iterdir():
        if child.is_dir() and (child / 'train').is_dir():
            return child
    return None


def _extract(zip_path, workdir):
    workdir = Path(workdir)
    workdir.mkdir(parents=True, exist_ok=True)
    print('Extracting', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(workdir)
    return workdir


def _copy(src, dst_dir):
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst_dir / src.name)


def prepare_dataset(src, dst):
    dst = Path(dst)
    base = _locate_base(src)
    if base is None:
        raise FileNotFoundError('No chest_xray/train|test layout under %s' % src)
    print('Using source:', base)
    for split in ['train', 'test']:
        split_dir = base / split
        if not split_dir.is_dir():
            print('  [skip]', split)
            continue
        normal = split_dir / 'NORMAL'
        n = 0
        if normal.is_dir():
            for f in normal.iterdir():
                if _is_img(f):
                    _copy(f, dst / 'stage1' / split / 'NORMAL')
                    n += 1
        bact, vir = _collect_pneumonia(split_dir / 'PNEUMONIA')
        for f in bact + vir:
            _copy(f, dst / 'stage1' / split / 'PNEUMONIA')
        for f in bact:
            _copy(f, dst / 'stage2' / split / 'BACTERIAL')
        for f in vir:
            _copy(f, dst / 'stage2' / split / 'VIRAL')
        print('  %s: NORMAL=%d PNEUMONIA=%d (BACTERIAL=%d, VIRAL=%d)'
              % (split, n, len(bact) + len(vir), len(bact), len(vir)))


def count_imgs(d):
    d = Path(d)
    return len([f for f in d.iterdir() if _is_img(f)]) if d.is_dir() else 0


REQUIRED = [
    DATA / 'stage1' / 'train' / 'NORMAL', DATA / 'stage1' / 'train' / 'PNEUMONIA',
    DATA / 'stage1' / 'test' / 'NORMAL', DATA / 'stage1' / 'test' / 'PNEUMONIA',
    DATA / 'stage2' / 'train' / 'BACTERIAL', DATA / 'stage2' / 'train' / 'VIRAL',
    DATA / 'stage2' / 'test' / 'BACTERIAL', DATA / 'stage2' / 'test' / 'VIRAL',
]
missing = [p for p in REQUIRED if not p.is_dir()]

if missing:
    print('Missing stage folders:', [str(p) for p in missing])
    candidates = [
        Path(PROJECT_DRIVE_DIR) / 'chest_xray',
        Path('/content/drive/MyDrive/chest_xray'),
        Path(PROJECT_DRIVE_DIR) / 'chest_xray.zip',
        Path('/content/drive/MyDrive/chest_xray.zip'),
    ]
    src = next((c for c in candidates if c.exists()), None)
    if src is None:
        raise SystemExit(
            'Dataset not found on Drive.\n'
            'Upload the raw chest_xray folder (or chest_xray.zip) to\n'
            '  %s\n'
            'then re-run this cell.' % PROJECT_DRIVE_DIR
        )
    if src.suffix.lower() == '.zip':
        src = _extract(src, '/content/_dataset')
    prepare_dataset(src, DATA)
else:
    print('Dataset already prepared on Drive.')

for split in ['train', 'test']:
    print('stage1/%s:' % split, {c: count_imgs(DATA / 'stage1' / split / c) for c in ['NORMAL', 'PNEUMONIA']})
    print('stage2/%s:' % split, {c: count_imgs(DATA / 'stage2' / split / c) for c in ['BACTERIAL', 'VIRAL']})


## 6. Build a fast 224px cache (local disk, persisted to Drive)

The raw X-rays are large. Decoding them every epoch is the main CPU bottleneck.
The cache is built once, reused across sessions, and copied to local disk for speed.


In [ ]:
import shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from PIL import Image

CACHE = Path('/content/cache224')
CACHE_DRIVE = Path(PROJECT_DRIVE_DIR) / 'cache224'
SIZE = 224
RESAMPLE = getattr(Image, 'Resampling', Image).BILINEAR
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}


def _resize_one(job):
    src, dst = job
    try:
        dst = Path(dst)
        dst.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(src) as im:
            im.convert('L').resize((SIZE, SIZE), RESAMPLE).save(dst, 'JPEG', quality=92)
        return 1
    except Exception as exc:
        print('  [warn]', src, exc)
        return 0


def build_cache(src_root, dst_root):
    src_root, dst_root = Path(src_root), Path(dst_root)
    if not src_root.is_dir():
        print('skip (missing):', src_root)
        return
    jobs = []
    for p in src_root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            out = dst_root / p.relative_to(src_root).with_suffix('.jpg')
            if not out.exists():
                jobs.append((str(p), str(out)))
    print('%s -> %d images to cache' % (src_root, len(jobs)))
    if not jobs:
        return
    with ThreadPoolExecutor(max_workers=4) as ex:
        done = sum(ex.map(_resize_one, jobs))
    print('  cached', done)


def _has_images(root):
    root = Path(root)
    return root.is_dir() and any(p.suffix.lower() == '.jpg' for p in root.rglob('*.jpg'))


if _has_images(CACHE):
    print('Local cache already present:', CACHE)
elif _has_images(CACHE_DRIVE):
    print('Copying cache from Drive...')
    shutil.copytree(CACHE_DRIVE, CACHE, dirs_exist_ok=True)
else:
    for stage in ['stage1', 'stage2']:
        for split in ['train', 'test']:
            build_cache(DATA / stage / split, CACHE / stage / split)
    print('Persisting cache to Drive (one-time)...')
    shutil.copytree(CACHE, CACHE_DRIVE, dirs_exist_ok=True)

STAGE1_TRAIN = str(CACHE / 'stage1' / 'train')
STAGE2_TRAIN = str(CACHE / 'stage2' / 'train')
STAGE1_TEST = str(CACHE / 'stage1' / 'test')
STAGE2_TEST = str(CACHE / 'stage2' / 'test')
print('Stage 1 train:', STAGE1_TRAIN)
print('Stage 2 train:', STAGE2_TRAIN)


## 7. Sanity check (tiny subset, ~1 min)

Confirms the data paths, GPU, class weighting and CLI before the long runs.


In [ ]:
import os
import random
import shutil

random.seed(42)

SANITY_DIR = '/content/sanity_data'
if not os.path.exists(os.path.join(SANITY_DIR, 'NORMAL')):
    for cls in ['NORMAL', 'PNEUMONIA']:
        src = os.path.join(STAGE1_TRAIN, cls)
        dst = os.path.join(SANITY_DIR, cls)
        os.makedirs(dst, exist_ok=True)
        files = os.listdir(src)
        for f in random.sample(files, min(32, len(files))):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))

!python -m pneumonia train --config configs/stage1_pneumonet_v1.yaml --data "{SANITY_DIR}" --checkpoint-dir /content/sanity_ckpt --num-workers 0

print('\nSanity run finished.')


## 8. Train Stage 1 (detection)

Order: PneumoNet-v1 (already trained → skipped) → PneumoNet-v2 → VGG-16 → Inception-v3.

Checkpoints go to local disk; `final_model.pt` is copied to Drive on success.


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

CKPT_DRIVE = Path(CKPT_DRIVE)
LOCAL_CKPT = Path('/content/ckpt')
LOCAL_CKPT.mkdir(parents=True, exist_ok=True)


def train_one(cfg_name, data_dir):
    drive_dir = CKPT_DRIVE / cfg_name
    final_drive = drive_dir / 'final_model.pt'
    if final_drive.exists():
        print('[SKIP] %s already trained (%.0f MB on Drive)' % (cfg_name, final_drive.stat().st_size / 1e6))
        return True

    local_dir = LOCAL_CKPT / cfg_name
    local_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, '-m', 'pneumonia', 'train',
        '--config', 'configs/%s.yaml' % cfg_name,
        '--data', str(data_dir),
        '--checkpoint-dir', str(local_dir),
        '--num-workers', '2',
    ]
    print('\n' + '=' * 70)
    print(' '.join(cmd))
    print('=' * 70, flush=True)
    rc = subprocess.call(cmd)
    if rc != 0:
        print('[FAILED] %s exited with code %d' % (cfg_name, rc))
        return False

    final_local = local_dir / 'final_model.pt'
    if not final_local.exists():
        print('[FAILED] %s produced no final_model.pt' % cfg_name)
        return False

    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(final_local, final_drive)
    print('[SAVED] %s (%.0f MB)' % (final_drive, final_drive.stat().st_size / 1e6))

    # Epoch checkpoints carry optimizer state (hundreds of MB) - not needed once done.
    for p in local_dir.glob('checkpoint_epoch*.pt'):
        p.unlink()
    return True


STAGE1_JOBS = [
    'stage1_pneumonet_v1',
    'stage1_pneumonet_v2',
    'stage1_vgg16',
    'stage1_inception',
]

for cfg in STAGE1_JOBS:
    if not train_one(cfg, STAGE1_TRAIN):
        print('\nStopping Stage 1. Fix the error above, then re-run this cell:')
        print('completed models are skipped automatically.')
        break
else:
    print('\nStage 1 complete.')


## 9. Train Stage 2 (bacterial vs viral)

Stage 2 is independent of Stage 1 — it trains only on pneumonia images.


In [ ]:
STAGE2_JOBS = [
    'stage2_pneumonet_v1',
    'stage2_mobilenet',
    'stage2_xception',
]

for cfg in STAGE2_JOBS:
    if not train_one(cfg, STAGE2_TRAIN):
        print('\nStopping Stage 2. Fix the error above, then re-run this cell.')
        break
else:
    print('\nStage 2 complete.')


## 10. Evaluate every model individually

In [ ]:
for cfg in STAGE1_JOBS:
    ckpt = CKPT_DRIVE / cfg / 'final_model.pt'
    if not ckpt.exists():
        print('[SKIP] %s: no checkpoint' % cfg)
        continue
    print('\n' + '=' * 70)
    print('Evaluating %s on the Stage 1 test set' % cfg)
    print('=' * 70)
    !python -m pneumonia evaluate --config configs/{cfg}.yaml --data "{STAGE1_TEST}" --checkpoint "{ckpt}" --num-workers 2

for cfg in STAGE2_JOBS:
    ckpt = CKPT_DRIVE / cfg / 'final_model.pt'
    if not ckpt.exists():
        print('[SKIP] %s: no checkpoint' % cfg)
        continue
    print('\n' + '=' * 70)
    print('Evaluating %s on the Stage 2 test set' % cfg)
    print('=' * 70)
    !python -m pneumonia evaluate --config configs/{cfg}.yaml --data "{STAGE2_TEST}" --checkpoint "{ckpt}" --num-workers 2


## 11. Wire up the evaluation scripts

The ensemble scripts read relative paths (`results/...`, `data/real_3class/test`,
`data/real_stage2/test`). This cell links those to the Drive checkpoints and the
224px cache, including the 3-class test set needed for the full pipeline.


In [ ]:
import os
from pathlib import Path

# --- results/ -> Drive checkpoints -----------------------------------------
local_results = Path('results')
local_results.mkdir(parents=True, exist_ok=True)

ALL_CFGS = STAGE1_JOBS + STAGE2_JOBS
for cfg in ALL_CFGS:
    link = local_results / cfg
    target = Path(CKPT_DRIVE) / cfg
    if not target.exists():
        print('[missing]', target)
        continue
    if link.exists() or link.is_symlink():
        link.unlink()
    link.symlink_to(target, target_is_directory=True)

# evaluate_stage2_ensemble.py expects stage2_checkpoints/ for PneumoNet-v1
alias = local_results / 'stage2_checkpoints'
if alias.exists() or alias.is_symlink():
    alias.unlink()
alias.symlink_to(Path(CKPT_DRIVE) / 'stage2_pneumonet_v1', target_is_directory=True)

# --- test data bridges ------------------------------------------------------
def link_dir(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.is_dir():
        print('[missing]', src)
        return
    if dst.exists() or dst.is_symlink():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.symlink_to(src, target_is_directory=True)


three = Path('data') / 'real_3class' / 'test'
link_dir(Path(STAGE1_TEST) / 'NORMAL', three / 'NORMAL')
link_dir(Path(STAGE2_TEST) / 'BACTERIAL', three / 'BACTERIAL')
link_dir(Path(STAGE2_TEST) / 'VIRAL', three / 'VIRAL')
link_dir(Path(STAGE2_TEST), Path('data') / 'real_stage2' / 'test')

print('Evaluation layout ready.')
for p in sorted(local_results.iterdir()):
    print('  results/' + p.name)


## 12. Stage 1 ensemble — paper weights (VGG-16:5, Inception-v3:4, PneumoNet-v2:3, PneumoNet-v1:1)

In [ ]:
!python scripts/evaluate_stage1_ensemble.py


## 13. Stage 2 ensemble — weights found by grid search

In [ ]:
!python scripts/evaluate_stage2_ensemble.py


## 14. Full two-stage pipeline (Normal → Bacterial / Viral)

In [ ]:
!python scripts/evaluate_full_pipeline.py


## 15. Artifacts on Drive

In [ ]:
from pathlib import Path

print('Trained models on Drive:')
for p in sorted(Path(CKPT_DRIVE).rglob('final_model.pt')):
    print('  %-70s %6.1f MB' % (p, p.stat().st_size / 1e6))

print('\nData:', DATA_DRIVE)
print('Cache:', Path(PROJECT_DRIVE_DIR) / 'cache224')


## Resume / troubleshooting

**Session disconnected mid-model**
Re-run cells 1–6, then jump to cell 8 (Stage 1) or 9 (Stage 2). Models with a
`final_model.pt` on Drive are skipped; only the interrupted model retrains.

**`No module named pneumonia`**
Cell 3 must be run (`pip install -e .`). If you re-cloned, run it again.

**CUDA out of memory**
Lower `data.batch_size` in the relevant `configs/*.yaml` (e.g. 32 → 16) and re-run
that model. `--no-amp` is available for debugging, but AMP is what keeps T4 fast.

**"AMP present in repo: False"**
The GitHub repo does not contain the speedups yet. Commit and push `src/`, then
re-run cells 2–4.

**Slow epochs**
Check that the 224px cache was used (`Stage 1 train: /content/cache224/...`) and
that `workers=2` appears in the CLI banner.
